In [1]:
import pandas as pd, numpy as np, glob, os
from pathlib import Path

ROOT = Path(r"C:\Datasets\EdNet")
Q_PATH = ROOT / "contents" / "questions.csv"
KT1_DIR = ROOT / "KT1"
OUT = Path(r"C:\Datasets\processed")      # nơi lưu kết quả trung gian
OUT.mkdir(exist_ok=True)

q = pd.read_csv(Q_PATH)
print(q.shape)
print(q.columns.tolist())
print(q.head())
print(q["part"].value_counts())

(13169, 7)
['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']
  question_id bundle_id explanation_id correct_answer  part          tags  \
0          q1        b1             e1              b     1   1;2;179;181   
1          q2        b2             e2              a     1      15;2;182   
2          q3        b3             e3              b     1  14;2;179;183   
3          q4        b4             e4              b     1   9;2;179;184   
4          q5        b5             e5              c     1   8;2;179;181   

     deployed_at  
0  1558093217098  
1  1558093219720  
2  1558093222784  
3  1558093225357  
4  1558093228439  
part
5    5703
2    1662
7    1402
6    1335
3    1266
4    1158
1     643
Name: count, dtype: int64


In [ ]:
q["tag_list"] = q["tags"].astype(str).str.split(";")
q_matrix = (q[["question_id", "tag_list"]]
            .explode("tag_list")
            .assign(v=1)
            .pivot_table(index="question_id", columns="tag_list",
                         values="v", fill_value=0))
print(q_matrix.shape)
q_matrix.to_parquet(OUT / "q_matrix.parquet")

In [ ]:
files = glob.glob(str(KT1_DIR / "*.csv"))
print("Số file KT1:", len(files))

rng = np.random.default_rng(42)
sample = rng.choice(files, size=5000, replace=False)

dfs = []
for f in sample:
    d = pd.read_csv(f)
    d["user_id"] = Path(f).stem
    dfs.append(d)
log = pd.concat(dfs, ignore_index=True)

log = log.merge(q[["question_id", "correct_answer"]], on="question_id", how="left")
log["correct"] = (log["user_answer"] == log["correct_answer"]).astype(int)
print(log.shape, "tỷ lệ đúng:", log["correct"].mean().round(3))
print("Câu thiếu đáp án đúng:", log["correct_answer"].isna().sum())

In [ ]:
log = log.sort_values("timestamp")
first = log.drop_duplicates(["user_id", "question_id"], keep="first")
first.to_parquet(OUT / "first_attempts_5000.parquet")
print(first.shape)

In [ ]:
# kiểm tra kiểu của cột part: số (5) hay chuỗi ("5") rồi chỉnh dòng dưới nếu cần
p5 = q.loc[q["part"] == 5, "question_id"]
first5 = first[first["question_id"].isin(p5)]

N_ITEMS, MIN_ANS = 200, 50
top_items = first5["question_id"].value_counts().head(N_ITEMS).index
block = first5[first5["question_id"].isin(top_items)]

cnt = block["user_id"].value_counts()
block = block[block["user_id"].isin(cnt[cnt >= MIN_ANS].index)]

R = block.pivot(index="user_id", columns="question_id", values="correct")
print(R.shape, "tỷ lệ ô có dữ liệu:", R.notna().mean().mean().round(3))
R.to_parquet(OUT / "R_part5.parquet")

In [ ]:
p = R.mean()                                   # độ khó (tỷ lệ đúng)

# điểm "phần còn lại" (bỏ chính câu đang xét), dùng tỷ lệ đúng
# vì mỗi người trả lời số câu khác nhau
n_ans = R.notna().sum(axis=1)
sum_ans = R.sum(axis=1)
def item_rest_corr(col):
    rest = (sum_ans - col.fillna(0)) / (n_ans - col.notna().astype(int))
    return col.corr(rest)
r_pb = R.apply(item_rest_corr)

ctt = pd.DataFrame({"p": p, "r_pb": r_pb, "n": R.notna().sum()})
print(ctt.describe())
print(ctt[(ctt.p > 0.95) | (ctt.p < 0.10) | (ctt.r_pb < 0.10)])

In [10]:
print("== Câu hỏi ==")
print(q.columns.tolist())
print(q["part"].value_counts())
print("Q-matrix:", q_matrix.shape)

print("\n== Log ==")
print("Số file KT1:", len(files))
print("log:", log.shape, "| thiếu đáp án đúng:", log["correct_answer"].isna().sum())
print("first (lần làm đầu):", first.shape, "| số người:", first["user_id"].nunique())

print("\n== Khối IRT (Part 5) ==")
print("R:", R.shape, "| tỷ lệ ô có dữ liệu:", R.notna().mean().mean().round(3))
print("Số câu mỗi người (trung vị):", R.notna().sum(axis=1).median())
print("Số người mỗi câu (nhỏ nhất / trung vị):",
      R.notna().sum().min(), "/", R.notna().sum().median())

print("\n== CTT ==")
print(ctt.describe().round(3))
print("Số câu cần xem lại:",
      ((ctt.p > 0.95) | (ctt.p < 0.10) | (ctt.r_pb < 0.10)).sum())

== Câu hỏi ==
['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at', 'tag_list']
part
5    5703
2    1662
7    1402
6    1335
3    1266
4    1158
1     643
Name: count, dtype: int64
Q-matrix: (13169, 189)

== Log ==
Số file KT1: 784309
log: (652345, 8) | thiếu đáp án đúng: 0
first (lần làm đầu): (543375, 8) | số người: 5000

== Khối IRT (Part 5) ==
R: (273, 200) | tỷ lệ ô có dữ liệu: 0.482
Số câu mỗi người (trung vị): 85.0
Số người mỗi câu (nhỏ nhất / trung vị): 22 / 129.5

== CTT ==
             p     r_pb        n
count  200.000  200.000  200.000
mean     0.519    0.187  131.700
std      0.156    0.129   42.915
min      0.109   -0.336   22.000
25%      0.418    0.106  110.000
50%      0.522    0.189  129.500
75%      0.643    0.278  161.000
max      0.856    0.482  255.000
Số câu cần xem lại: 44
